<a href="https://colab.research.google.com/github/madandivvela/currencyagent/blob/main/action_autonomy.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Setup for Colab vs Local
import os
import sys
import zipfile
import glob

# Check if running on Colab
IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    # Unzip the uploaded course files
    zip_path = '/content/agentic-ai-build-your-first-agentic-ai-system-4645038.zip'

    if os.path.exists(zip_path):
        print("Extracting course files...")
        with zipfile.ZipFile(zip_path, 'r') as zip_ref:
            zip_ref.extractall('/content/')
        print("✓ Course files extracted")
    else:
        raise FileNotFoundError(
            f"Please upload the course ZIP file to Colab first.\n"
            f"Expected location: {zip_path}\n"
            f"How to upload: Drag and drop the ZIP file into the Files panel (📁) on the left"
        )

    # Find the extracted directory (handles different ZIP structures)
    # Look for directories that contain 'assets' and 'data' folders
    possible_dirs = glob.glob('/content/agentic-ai-build-*/')
    course_dir = None

    for dir_path in possible_dirs:
        if os.path.exists(os.path.join(dir_path, 'assets')) and os.path.exists(os.path.join(dir_path, 'data')):
            course_dir = dir_path
            break

    if not course_dir:
        raise FileNotFoundError(
            "Could not find course directory after extraction.\n"
            "Please ensure the ZIP contains 'assets' and 'data' folders."
        )

    # Navigate to course directory
    os.chdir(course_dir)
    print(f"✓ Working directory: {os.getcwd()}")

    # Get API key from Colab secrets
    from google.colab import userdata
    os.environ['OPENAI_API_KEY'] = userdata.get('OPENAI_API_KEY')
    print("✓ API key loaded from Colab secrets")
else:
    # Local environment - use .env file
    from dotenv import load_dotenv
    load_dotenv()

# Verify API key is set
if not os.getenv('OPENAI_API_KEY'):
    raise ValueError("Please set OPENAI_API_KEY in Colab Secrets or .env file")

print("\n" + "="*60)
print("Environment setup complete!")
print("="*60)

In [4]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


# V1: Action Autonomy - Router Agent

## The Autonomy Ladder

Building effective AI agents requires a deliberate approach to increasing autonomy:



**Key Philosophy:** Start with a narrow, well-defined scope. Validate thoroughly. Then expand deliberately.

## What is Action Autonomy?

**Definition:** Agent performs single, well-defined classification or routing actions.

**Use Case:** Customer support routing
- Input: Customer message
- Action: Classify intent and route to department
- Output: Routing decision
- Handoff: Human agent takes over




In [5]:
from IPython.display import Image, display
display(Image('/content/agentic-ai-build-your-first-agentic-ai-system-4645038/assets/diagrams/autonomy_ladder.png', width=600))

FileNotFoundError: No such file or directory: '/content/agentic-ai-build-your-first-agentic-ai-system-4645038/assets/diagrams/autonomy_ladder.png'

FileNotFoundError: No such file or directory: '/content/agentic-ai-build-your-first-agentic-ai-system-4645038/assets/diagrams/autonomy_ladder.png'

<IPython.core.display.Image object>

## Setup

Install required packages and set up environment.

In [6]:
# Install packages
!pip install -q openai pandas python-dotenv
!pip install -q 'arize-phoenix[evals]' openinference-instrumentation-openai

print("Packages installed successfully!")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.9/40.9 kB 2.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.7/52.7 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 324.5/324.5 kB 13.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 185.8/185.8 kB 12.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 121.4/121.4 kB 8.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.5/40.5 kB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 928.0/928.0 kB 31.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 80.1/80.1 kB 5.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 82.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.3/5.3 MB 107.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.8/67.8 kB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 432.2/432.2 kB 26.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.

## Building the Router Agent

### Architecture

Our V1 agent has a simple 4-step process:





### Key Design Choices

1. **Model:** GPT-4o-mini (cost-effective for classification)
2. **Temperature:** 0.1 (consistent results)
3. **Output:** JSON mode (structured response)
4. **Fallback:** ESCALATION if invalid department

Let's build it step by step.

In [7]:
from IPython.display import Image, display

print("V1 Router Architecture:")
display(Image('/content/agentic-ai-build-your-first-agentic-ai-system-4645038/assets/diagrams/v1_architecture.png', width=600))

print("\nData Flow Through System:")
display(Image('/content/agentic-ai-build-your-first-agentic-ai-system-4645038/assets/diagrams/v1_data_flow.png', width=600))

V1 Router Architecture:


FileNotFoundError: No such file or directory: '/content/agentic-ai-build-your-first-agentic-ai-system-4645038/assets/diagrams/v1_architecture.png'

FileNotFoundError: No such file or directory: '/content/agentic-ai-build-your-first-agentic-ai-system-4645038/assets/diagrams/v1_architecture.png'

<IPython.core.display.Image object>


Data Flow Through System:


FileNotFoundError: No such file or directory: '/content/agentic-ai-build-your-first-agentic-ai-system-4645038/assets/diagrams/v1_data_flow.png'

FileNotFoundError: No such file or directory: '/content/agentic-ai-build-your-first-agentic-ai-system-4645038/assets/diagrams/v1_data_flow.png'

<IPython.core.display.Image object>

In [8]:
# Step 1: Define data structures

from enum import Enum
from dataclasses import dataclass

class Department(Enum):
    """Available departments for routing."""
    BILLING = "billing"
    RETURNS = "returns"
    TECHNICAL_SUPPORT = "technical_support"
    ORDER_STATUS = "order_status"
    PRODUCT_INQUIRY = "product_inquiry"
    ACCOUNT_MANAGEMENT = "account_management"
    ESCALATION = "escalation"

@dataclass
class RoutingDecision:
    """Result of routing decision."""
    department: Department
    reasoning: str
    customer_message: str

print("Data structures defined!")
print(f"\nAvailable departments: {[d.name for d in Department]}")

Data structures defined!

Available departments: ['BILLING', 'RETURNS', 'TECHNICAL_SUPPORT', 'ORDER_STATUS', 'PRODUCT_INQUIRY', 'ACCOUNT_MANAGEMENT', 'ESCALATION']


In [9]:
# Step 2: Define Prompt 1 (baseline)

# Starting with minimal prompt - no department descriptions
# We'll discover what's missing through evaluation

SYSTEM_PROMPT_1 = """Route customer messages to departments.

Available departments: BILLING, RETURNS, TECHNICAL_SUPPORT, ORDER_STATUS, PRODUCT_INQUIRY, ACCOUNT_MANAGEMENT, ESCALATION

Respond with JSON:
{
    "department": "DEPARTMENT_NAME",
    "reasoning": "Your reasoning"
}
"""

print("Prompt 1 (baseline) defined!")
print(f"Prompt length: {len(SYSTEM_PROMPT_1)} chars")
print("\nNote: This is intentionally minimal. We'll see what happens...")

Prompt 1 (baseline) defined!
Prompt length: 258 chars

Note: This is intentionally minimal. We'll see what happens...


In [10]:
# Step 3: Build the RouterAgent class

import json
from openai import OpenAI

class RouterAgent:
    """V1 Action Autonomy Agent - Routes customer messages to departments."""

    def __init__(self, system_prompt):
        """Initialize agent with a system prompt."""
        self.client = OpenAI(api_key=os.getenv('OPENAI_API_KEY'))
        self.model = "gpt-4o-mini"
        self.system_prompt = system_prompt

    def route(self, customer_message: str) -> RoutingDecision:
        """Route a customer message to appropriate department."""

        # Step 1: Call OpenAI API
        response = self.client.chat.completions.create(
            model=self.model,
            messages=[
                {"role": "system", "content": self.system_prompt},
                {"role": "user", "content": customer_message}
            ],
            temperature=0.1,
            response_format={"type": "json_object"}
        )

        # Step 2: Parse JSON response
        result = json.loads(response.choices[0].message.content)

        # Step 3: Validate department
        dept_name = result.get("department", "ESCALATION").upper()
        try:
            department = Department[dept_name]
        except KeyError:
            department = Department.ESCALATION

        # Step 4: Return structured decision
        return RoutingDecision(
            department=department,
            reasoning=result.get("reasoning", "No reasoning provided"),
            customer_message=customer_message
        )

print("RouterAgent class defined!")
print("Ready to route customer messages.")

RouterAgent class defined!
Ready to route customer messages.


## Demo: See the Agent in Action

Let's test our agent with a few examples before formal evaluation.

In [11]:
# Initialize agent with Prompt 1
agent = RouterAgent(system_prompt=SYSTEM_PROMPT_1)

# Test messages covering different departments
test_messages = [
    "I was charged twice for my order!",
    "Where is my package? It's been 2 weeks!",
    "I want to return these shoes, they don't fit",
    "Is the blue wireless headphone in stock?",
    "I can't log into my account, it says password invalid",
    "This is ridiculous! I've called 3 times and nobody helps me!"
]

print("=" * 70)
print("ROUTER AGENT DEMO (Prompt 1 Baseline)")
print("=" * 70)
print()

for i, message in enumerate(test_messages, 1):
    print(f"[{i}] Customer: {message}")

    decision = agent.route(message)

    print(f"    -> Department: {decision.department.name}")
    print(f"    -> Reasoning: {decision.reasoning}")
    print()

print("Demo looks good! But let's evaluate systematically...")

OpenAIError: Missing credentials. Please pass an `api_key`, `workload_identity`, `admin_api_key`, or set the `OPENAI_API_KEY` or `OPENAI_ADMIN_KEY` environment variable.

---

## 🎬 End of Chapter

---

## Evaluation Setup

### Why Evaluate?

Demo showed it works, but we need systematic evaluation:
- Does it handle edge cases?
- What's the accuracy across all departments?
- Where does it fail and why?

### Evaluation Metric: Routing Accuracy

For Prompt 2 (Action Autonomy), routing accuracy is the right metric:
- **Clear ground truth:** Each message has one correct department
- **Binary outcome:** Either correct or incorrect
- **Easy to interpret:** 85% accuracy means 85% of routings are correct

### Test Dataset

30 test cases covering:
- All 7 departments
- Simple cases (clear keywords)
- Ambiguous cases (multiple possible departments)
- Edge cases (unusual requests)

### Evaluation Workflow



In [ ]:
# Load test cases
import pandas as pd

# Load from repository data directory
test_df = pd.read_csv('/content/agentic-ai-build-your-first-agentic-ai-system-4645038/data/v1_test_cases.csv')

print(f"Loaded {len(test_df)} test cases")
print(f"\nColumns: {list(test_df.columns)}")
print(f"\nDepartment distribution:")
print(test_df['expected_department'].value_counts())

# Show a few examples
print(f"\nSample test cases:")
print(test_df[['test_id', 'customer_message', 'expected_department', 'category']].head())

## Setup Arize Phoenix for Observability

### Why Phoenix?

Phoenix captures every LLM call as a "trace":
- Input: Customer message
- Prompt: System prompt sent to LLM
- Output: Department and reasoning
- Metadata: Tokens, latency, cost

This lets us:
1. See exactly what the agent is thinking
2. Understand why failures happen
3. Identify patterns in errors
4. Make targeted improvements

In [ ]:
# Start Phoenix (Colab-compatible setup)
import os

# Configure Phoenix for Colab/local compatibility
os.environ["PHOENIX_HOST"] = "0.0.0.0"
os.environ["PHOENIX_PORT"] = "6006"

import phoenix as px
from phoenix.otel import register
from openinference.instrumentation.openai import OpenAIInstrumentor

print("Starting Arize Phoenix...")
session = px.launch_app()  # don't pass port parameter
print("Phoenix session url:", session.url)

# For Google Colab compatibility
try:
    from google.colab import output
    output.serve_kernel_port_as_window(6006)
    print("✓ Phoenix running on Colab at port 6006")
except ImportError:
    print("✓ Phoenix running locally at http://localhost:6006")

print("\nClick the link above to open Phoenix UI in a new tab.")
print("Keep this tab open while running evaluations.")

## Run Prompt 1 Evaluation

Let's evaluate the baseline (Prompt 1) to establish our starting point.

In [ ]:
# Enable tracing for Prompt 1
project_name = "V1_action_autonomy_prompt_1"
print(f"Enabling tracing for project: {project_name}")

tracer_provider = register(project_name=project_name)
OpenAIInstrumentor().instrument(tracer_provider=tracer_provider)

print("Tracing enabled! All API calls will be captured in Phoenix.")

In [ ]:
# Run Prompt 1 evaluation
from dataclasses import dataclass
from collections import defaultdict
from opentelemetry import trace
from opentelemetry.trace import Status, StatusCode

@dataclass
class EvalResult:
    """Result of a single evaluation."""
    test_id: str
    message: str
    expected: str
    predicted: str
    correct: bool
    reasoning: str
    category: str

# Initialize agent with Prompt 1
agent_p1 = RouterAgent(system_prompt=SYSTEM_PROMPT_1)
tracer = trace.get_tracer(__name__)

results_p1 = []

print("Running Prompt 1 evaluation on 30 test cases...")
print("(Each routing decision is being traced in Phoenix)\n")

for idx, row in test_df.iterrows():
    i = idx + 1
    test_id = row['test_id']

    # Create custom span for better Phoenix visualization
    with tracer.start_as_current_span(f"test_case_{test_id}") as span:
        span.set_attribute("test.id", test_id)
        span.set_attribute("test.category", row['category'])
        span.set_attribute("test.expected_department", row['expected_department'])

        # Route the message
        decision = agent_p1.route(row['customer_message'])
        correct = decision.department.name == row['expected_department']

        # Record result in span
        span.set_attribute("result.predicted_department", decision.department.name)
        span.set_attribute("result.correct", correct)

        if correct:
            span.set_status(Status(StatusCode.OK))
        else:
            span.set_status(Status(StatusCode.ERROR, "Incorrect routing"))
            span.set_attribute("error.expected", row['expected_department'])
            span.set_attribute("error.got", decision.department.name)

        # Store result
        result = EvalResult(
            test_id=test_id,
            message=row['customer_message'],
            expected=row['expected_department'],
            predicted=decision.department.name,
            correct=correct,
            reasoning=decision.reasoning,
            category=row['category']
        )
        results_p1.append(result)

        # Show progress
        status = "PASS" if correct else "FAIL"
        print(f"[{i}/30] {test_id}: {status} (Expected: {result.expected}, Got: {result.predicted})")

print("\nEvaluation complete!")

In [ ]:
# Compute Prompt 1 metrics
total = len(results_p1)
correct = sum(1 for r in results_p1 if r.correct)
accuracy = correct / total

# Per-department accuracy
dept_correct = defaultdict(int)
dept_total = defaultdict(int)
for r in results_p1:
    dept_total[r.expected] += 1
    if r.correct:
        dept_correct[r.expected] += 1

print("=" * 70)
print("PROMPT 1 EVALUATION RESULTS")
print("=" * 70)

print(f"\nOverall Accuracy: {accuracy:.1%} ({correct}/{total} correct)")

print(f"\nPer-Department Accuracy:")
for dept in sorted(dept_total.keys()):
    acc = dept_correct[dept] / dept_total[dept]
    bar = "█" * int(acc * 20) + "░" * (20 - int(acc * 20))
    print(f"  {dept:20} {bar} {acc:.0%}")

# Show errors
errors = [r for r in results_p1 if not r.correct]
if errors:
    print(f"\nErrors ({len(errors)} cases):")
    for r in errors:
        print(f"\n  [{r.test_id}] {r.message[:60]}...")
        print(f"  Expected: {r.expected} -> Got: {r.predicted}")
        print(f"  Category: {r.category}")

---

## 🎬 End of Chapter

---

---

# 📊 Continuous Calibration (CC) Phase

**Goal:** Understand WHY the system fails and design metrics to measure performance.

**In this phase:**
- Observe failures in Phoenix traces
- Analyze error patterns
- Design evaluation metrics
- Identify root causes

**Output:** Clear understanding of what to fix and how to measure it.

---

## Analyze Failures in Arize Phoenix

Now comes the key part: **Understanding WHY failures happened**

### How to Use Phoenix

1. Open the Phoenix URL from above
2. Click "Traces" in the left sidebar
3. Select project "V1_action_autonomy_prompt_1"
4. Filter for failed cases (red status)
5. Click on each trace to see:
   - Customer message
   - System prompt sent to LLM
   - LLM's response (department + reasoning)
   - Why it was incorrect

### Common Failure Patterns

Look for patterns like:
- **Ambiguous keywords:** "refund" could be BILLING or RETURNS
- **Multi-issue messages:** Customer mentions both shipping and refund
- **Missing context:** Prompt 1 lacks department descriptions
- **Over-escalation:** Negative sentiment triggers ESCALATION unnecessarily

**Exercise:** Analyze 3-5 failed traces and note patterns you observe.


---

## 🎬 End of Chapter


---

---

# 🚀 Continuous Deployment (CD) Phase

**Goal:** Improve the system based on CC insights and measure impact.

**In this phase:**
- Make targeted improvements (Prompt 2)
- Re-evaluate with same metrics
- Compare before/after performance
- Validate improvements worked

**Output:** Better system with measured improvements.

---

## Improve to Prompt 2

Based on Phoenix analysis, we identified these issues in Prompt 1:

1. **No department descriptions** → LLM guesses based on keywords alone
2. **Ambiguous boundaries** → "refund status" routed to RETURNS instead of BILLING
3. **Password resets** → Routed to ACCOUNT_MANAGEMENT instead of TECHNICAL_SUPPORT

### V1 Improvements

The Prompt 2 adds:
- Clear descriptions for each department
- Explicit disambiguation rules
- Examples of edge cases

Let's see if it helps!

### The Iterative Improvement Cycle



In [ ]:
# Now let's create Prompt 2 with improvements based on what we learned

SYSTEM_PROMPT_2 = """Route customer messages to departments.

Available departments:
- BILLING: Payment issues, charges, refunds, refund status, account balances, fees
- RETURNS: Return requests, exchanges, return status, return policies
- TECHNICAL_SUPPORT: Login problems, password reset issues, website errors, checkout failures
- ORDER_STATUS: Order tracking, shipping updates, delivery questions, missing items
- PRODUCT_INQUIRY: Product questions, specifications, availability, pricing
- ACCOUNT_MANAGEMENT: Profile updates, changing saved payment methods, preferences, address changes
- ESCALATION: Very upset customers demanding managers, supervisor requests

Important:
- Login/password problems = TECHNICAL_SUPPORT (not ACCOUNT_MANAGEMENT)
- Updating payment methods = ACCOUNT_MANAGEMENT (not BILLING)
- Refund status = BILLING (not RETURNS)

Respond with JSON:
{
    \"department\": \"DEPARTMENT_NAME\",
    \"reasoning\": \"Your reasoning\"
}
"""

print("Prompt 2 (improved) created with improvements!")
print(f"\nPrompt 1 length: {len(SYSTEM_PROMPT_1)} chars")
print(f"Prompt 2 length: {len(SYSTEM_PROMPT_2)} chars")
print(f"\nAdded {len(SYSTEM_PROMPT_2) - len(SYSTEM_PROMPT_1)} chars of context")

In [ ]:
# Enable tracing for Prompt 2 (separate project)

# Uninstrument previous tracer to avoid overwriting Prompt 1 traces
OpenAIInstrumentor().uninstrument()

project_name_p2 = "V1_action_autonomy_prompt_2"
print(f"Enabling tracing for project: {project_name_p2}")

tracer_provider_p2 = register(project_name=project_name_p2)
OpenAIInstrumentor().instrument(tracer_provider=tracer_provider_p2)

print("Tracing enabled for Prompt 2!")

In [ ]:
# Run Prompt 2 evaluation
agent_p2 = RouterAgent(system_prompt=SYSTEM_PROMPT_2)
results_p2 = []

print("Running Prompt 2 evaluation on 30 test cases...\n")

for idx, row in test_df.iterrows():
    i = idx + 1
    test_id = row['test_id']

    with tracer.start_as_current_span(f"test_case_{test_id}") as span:
        span.set_attribute("test.id", test_id)
        span.set_attribute("test.expected_department", row['expected_department'])

        decision = agent_p2.route(row['customer_message'])
        correct = decision.department.name == row['expected_department']

        span.set_attribute("result.correct", correct)

        if correct:
            span.set_status(Status(StatusCode.OK))
        else:
            span.set_status(Status(StatusCode.ERROR, "Incorrect routing"))
            span.set_attribute("error.expected", row['expected_department'])
            span.set_attribute("error.got", decision.department.name)

        result = EvalResult(
            test_id=test_id,
            message=row['customer_message'],
            expected=row['expected_department'],
            predicted=decision.department.name,
            correct=correct,
            reasoning=decision.reasoning,
            category=row['category']
        )
        results_p2.append(result)

        status = "PASS" if correct else "FAIL"
        print(f"[{i}/30] {test_id}: {status}")

print("\nPrompt 2 evaluation complete!")

In [ ]:
# Compare V0 vs V1
correct_v1 = sum(1 for r in results_p2 if r.correct)
accuracy_v1 = correct_v1 / len(results_p2)

print("=" * 70)
print("PROMPT 1 vs PROMPT 2 COMPARISON")
print("=" * 70)

print(f"\nOverall Accuracy:")
print(f"  Prompt 1: {accuracy:.1%} ({correct}/{total})")
print(f"  Prompt 2: {accuracy_v1:.1%} ({correct_v1}/{total})")
improvement = accuracy_v1 - accuracy
print(f"  Improvement: +{improvement:.1%}")

# Which errors got fixed?
v0_errors = {r.test_id for r in results_p1 if not r.correct}
v1_errors = {r.test_id for r in results_p2 if not r.correct}

fixed = v0_errors - v1_errors
still_failing = v0_errors & v1_errors

if fixed:
    print(f"\nFixed in Prompt 2 ({len(fixed)} cases):")
    for test_id in sorted(fixed):
        r = next(r for r in results_p1 if r.test_id == test_id)
        print(f"  [{test_id}] {r.message[:50]}...")

if still_failing:
    print(f"\nStill Failing ({len(still_failing)} cases):")
    for test_id in sorted(still_failing):
        r = next(r for r in results_p2 if r.test_id == test_id)
        print(f"  [{test_id}] {r.message[:50]}...")

## Key Takeaways

**V1 Action Autonomy:**
- Routes customer messages to departments (~93% accuracy)
- Simple classification → perfect for well-defined routing tasks
- Observability (Phoenix) revealed failure patterns → targeted improvements

**When to use Action Autonomy:** Single-action classification with clear categories

**Next:** V2 Planning Autonomy adds multi-step reasoning and document retrieval